# heteroscedasticity

> 对应代码：`EconometricsCode/code_in_notes/Chap.7/heteroscedasticity.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.7`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.7")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

文件头注释，标明文件名。

我们首先设置随机数种子以保证模拟结果可复现。

In [ ]:
%%stata
set seed 19880505

我们先删除可能已存在的同名程序 dgp，避免旧定义干扰；然后定义一个 rclass 类程序 dgp，用于生成模拟数据并返回估计结果。syntax 命令声明了三个可选参数：obs 为样本量（默认 100），b 为真实的斜率系数（默认 1），hetero 控制异方差强度（默认 0，即同方差基准情形）。

在程序内部，我们先清空内存并设定样本量为 obs，然后用 tempvar 声明临时变量名（避免与真实变量冲突），生成标准正态解释变量 x。关键的一步是条件标准差 sigma 的设定：sigma = 1 + hetero * |x|，当 hetero = 0 时误差项同方差，当 hetero > 0 时误差项的波动随 |x| 增大而增大，即存在异方差。最后按线性模型 y = 3 + b*x + u 生成因变量，其中 u ~ N(0, sigma^2)。

接下来用普通最小二乘法（OLS）估计 y 对 x 的回归，并把系数估计值和默认（同方差假设下的）标准误分别存入标量返回。随后我们再用 robust 选项重新估计同一回归：注意 robust 并不改变点估计，只是把标准误换成异方差稳健（heteroscedasticity-robust，即 White/Eicker-Huber）标准误。通过比较两种标准误在重复抽样中的表现，我们可以考察异方差对常规标准误推断的影响。

In [ ]:
%%stata
cap program drop dgp
program define dgp, rclass
	syntax [, obs(integer 100) b(real 1) hetero(real 0)]
	drop _all
	set obs `obs'
	tempvar x y sigma
	gen `x'=rnormal()
	gen `sigma'=1+`hetero'*abs(`x')
	gen `y'=3+`b'*`x'+rnormal(0,`sigma')
	quietly: reg `y' `x'
	return scalar b=_b[`x']
	return scalar se=_se[`x']
	quietly: reg `y' `x', robust
	return scalar r_b=_b[`x']
	return scalar r_se=_se[`x']
end

我们用 simulate 命令把上述 dgp 程序重复运行 2000 次（同方差基准情形，hetero 取默认值 0），收集每次回归的系数 b、常规标准误 se、稳健回归系数 rb 与稳健标准误 rse，随后用 su（summarize 的简写）查看这些量在整个抽样分布上的均值与离散程度。我们关注的重点是：在同方差下，常规标准误 se 的均值应当接近 b 的真实抽样标准差（即 b 的标准差），说明此时常规推断是有效的。

In [ ]:
%%stata
// multicolinearty, size
simulate b=r(b) se=r(se) rb=r(r_b) rse=r(r_se), reps(2000): dgp
su

最后我们令 hetero(1) 打开异方差，再重复 2000 次模拟并汇总。对比两组结果可以看到：在异方差下，OLS 点估计仍然无偏（b 与 rb 的均值都接近真实值 1），但常规标准误 se 的均值会系统性地偏离 b 的真实抽样标准差，从而导致 t 检验、置信区间等推断失真；而稳健标准误 rse 的均值仍与抽样标准差接近，说明稳健标准误在异方差下依然给出正确的推断。这正是我们在存在异方差时应当使用稳健标准误的 Monte Carlo 证据。

In [ ]:
%%stata
simulate b=r(b) se=r(se) rb=r(r_b) rse=r(r_se), reps(2000): dgp, hetero(1)
su